# Лабораторна робота №2 — Частина 2
## Наука про дані: Individual Household Electric Power Consumption

Датасет: https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

## 0. Імпорт бібліотек

In [ ]:
import timeit
import pandas as pd
import numpy as np
from scipy import stats

## 1. Завантаження та відкриття датасету

Завантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset.

In [ ]:
# Якщо файл вже завантажено вручну, вкажіть шлях тут:
DATASET_PATH = "household_power_consumption.txt"

t_start = timeit.default_timer()

df = pd.read_csv(
    DATASET_PATH,
    sep=";",
    low_memory=False,
    na_values="?",
)

t_load = timeit.default_timer() - t_start
print(f"Завантажено за {t_load:.2f} с. Розмір: {df.shape}")
display(df.head())

## 2. Data Cleaning

Здійснити data cleaning (заповнення пропусків, приведення типів тощо).

In [ ]:
def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищає датасет:
    - Об'єднує Date та Time у datetime
    - Приводить числові колонки до float
    - Заповнює пропуски медіаною
    """
    t0 = timeit.default_timer()
    df = df.copy()

    # Об'єднуємо дату і час
    df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
    df = df.drop(columns=["Date", "Time"])

    # Числові колонки
    num_cols = [
        "Global_active_power", "Global_reactive_power", "Voltage",
        "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3",
    ]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Пропуски → медіана
    before = df[num_cols].isna().sum().sum()
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    after = df[num_cols].isna().sum().sum()
    print(f"Заповнено пропусків: {before - after}")

    elapsed = timeit.default_timer() - t0
    print(f"Cleaning завершено за {elapsed:.2f} с")
    return df


df = clean_dataset(df)
display(df.dtypes)
display(df.head())

## 3. Вибірки

Окремими функціями сформувати вибірки.

### 3.1 Записи, у яких загальна активна споживана потужність перевищує 5 кВт

In [ ]:
def high_power(df: pd.DataFrame, threshold: float = 5.0) -> pd.DataFrame:
    """
    Обирає всі записи, у яких Global_active_power > threshold (кВт).
    """
    t0 = timeit.default_timer()
    result = df[df["Global_active_power"] > threshold].reset_index(drop=True)
    elapsed = timeit.default_timer() - t0
    print(f"Записи з GAP > {threshold} кВт: {len(result)} рядків | час: {elapsed:.4f} с")
    return result


display(high_power(df).head(10))

### 3.2 Записи з силою струму 19–20 А, де пральна машина + холодильник > бойлер + кондиціонер

In [ ]:
def current_range_appliance_filter(df: pd.DataFrame) -> pd.DataFrame:
    """
    Обирає записи з Global_intensity в [19, 20] А,
    для яких Sub_metering_2 (пральна машина + холодильник)
    більше за Sub_metering_3 (бойлер + кондиціонер).
    """
    t0 = timeit.default_timer()
    mask_current = df["Global_intensity"].between(19, 20)
    mask_appliance = df["Sub_metering_2"] > df["Sub_metering_3"]
    result = df[mask_current & mask_appliance].reset_index(drop=True)
    elapsed = timeit.default_timer() - t0
    print(f"Відфільтровано: {len(result)} рядків | час: {elapsed:.4f} с")
    return result


display(current_range_appliance_filter(df).head(10))

### 3.3 Випадкова вибірка 500 000 записів і середні величини груп споживання

In [ ]:
def random_sample_means(df: pd.DataFrame, n: int = 500_000, seed: int = 42) -> dict:
    """
    Обирає n записів без повторів та обчислює середні величини
    усіх 3-х груп споживання електричної енергії (Sub_metering_1/2/3).
    """
    t0 = timeit.default_timer()
    sample = df.sample(n=min(n, len(df)), replace=False, random_state=seed)
    means = {
        "Sub_metering_1 (кухня)": sample["Sub_metering_1"].mean(),
        "Sub_metering_2 (пральна + холодильник)": sample["Sub_metering_2"].mean(),
        "Sub_metering_3 (бойлер + кондиціонер)": sample["Sub_metering_3"].mean(),
    }
    elapsed = timeit.default_timer() - t0
    print(f"Вибірка {len(sample)} записів. Час: {elapsed:.4f} с")
    for k, v in means.items():
        print(f"  {k}: {v:.4f}")
    return means


random_sample_means(df)

### 3.4 Записи після 18:00 з >6 кВт/хв, де група 2 найбільша, кожен 3-й з першої половини + кожен 4-й з другої

In [ ]:
def evening_high_load_sample(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Обирає записи після 18:00 з GAP > 6 кВт.
    2. Серед них — де Sub_metering_2 є найбільшою серед усіх трьох груп.
    3. Повертає кожен 3-й запис з першої половини та кожен 4-й з другої.
    """
    t0 = timeit.default_timer()

    # Крок 1: після 18:00 та GAP > 6
    evening = df[(df["datetime"].dt.hour >= 18) & (df["Global_active_power"] > 6)]

    # Крок 2: Sub_metering_2 > Sub_metering_1 та Sub_metering_2 > Sub_metering_3
    dominant2 = evening[
        (evening["Sub_metering_2"] > evening["Sub_metering_1"]) &
        (evening["Sub_metering_2"] > evening["Sub_metering_3"])
    ].reset_index(drop=True)

    # Крок 3: розбиваємо навпіл
    mid = len(dominant2) // 2
    first_half  = dominant2.iloc[:mid].iloc[::3]   # кожен 3-й
    second_half = dominant2.iloc[mid:].iloc[::4]   # кожен 4-й

    result = pd.concat([first_half, second_half]).reset_index(drop=True)
    elapsed = timeit.default_timer() - t0
    print(f"Результат: {len(result)} рядків | час: {elapsed:.4f} с")
    return result


display(evening_high_load_sample(df).head(10))

## 4. Нормалізація та стандартизація датасету

Пронормувати та стандартизувати вибраний датасет.

In [ ]:
NUM_COLS = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3",
]

def normalize(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Min-Max нормалізація: значення → [0, 1]."""
    t0 = timeit.default_timer()
    df_norm = df.copy()
    for col in cols:
        mn, mx = df[col].min(), df[col].max()
        df_norm[col] = (df[col] - mn) / (mx - mn) if mx != mn else 0.0
    elapsed = timeit.default_timer() - t0
    print(f"Нормалізація завершена за {elapsed:.4f} с")
    return df_norm


def standardize(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Z-score стандартизація: (x - mean) / std."""
    t0 = timeit.default_timer()
    df_std = df.copy()
    for col in cols:
        mean, std = df[col].mean(), df[col].std()
        df_std[col] = (df[col] - mean) / std if std != 0 else 0.0
    elapsed = timeit.default_timer() - t0
    print(f"Стандартизація завершена за {elapsed:.4f} с")
    return df_std


df_normalized   = normalize(df, NUM_COLS)
df_standardized = standardize(df, NUM_COLS)

print("\nНормалізовані дані (перші рядки):")
display(df_normalized[NUM_COLS].head())
print("\nСтандартизовані дані (перші рядки):")
display(df_standardized[NUM_COLS].head())

## 5. Коефіцієнти кореляції Пірсона та Спірмена

Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [ ]:
def compute_correlations(df: pd.DataFrame, col1: str, col2: str) -> None:
    """
    Обчислює і виводить коефіцієнти Пірсона та Спірмена для двох колонок.
    """
    t0 = timeit.default_timer()
    pearson_r, pearson_p = stats.pearsonr(df[col1], df[col2])
    spearman_r, spearman_p = stats.spearmanr(df[col1], df[col2])
    elapsed = timeit.default_timer() - t0

    print(f"Кореляція між '{col1}' та '{col2}':")
    print(f"  Пірсон:  r = {pearson_r:.4f},  p = {pearson_p:.2e}")
    print(f"  Спірмен: r = {spearman_r:.4f}, p = {spearman_p:.2e}")
    print(f"  Час виконання: {elapsed:.4f} с")


compute_correlations(df, "Global_active_power", "Global_intensity")

## 6. One Hot Encoding категоріального атрибута

Провести One Hot Encoding категоріального атрибута.

In [ ]:
def one_hot_hour_period(df: pd.DataFrame) -> pd.DataFrame:
    """
    Створює категоріальний атрибут 'hour_period' (ніч/ранок/день/вечір)
    та застосовує One Hot Encoding.
    """
    t0 = timeit.default_timer()
    df = df.copy()

    def period(hour: int) -> str:
        if 0 <= hour < 6:
            return "ніч"
        elif 6 <= hour < 12:
            return "ранок"
        elif 12 <= hour < 18:
            return "день"
        else:
            return "вечір"

    df["hour_period"] = df["datetime"].dt.hour.map(period).astype("category")

    dummies = pd.get_dummies(df["hour_period"], prefix="period")
    df = pd.concat([df, dummies], axis=1)

    elapsed = timeit.default_timer() - t0
    print(f"OHE завершено за {elapsed:.4f} с. Нові колонки: {dummies.columns.tolist()}")
    return df


df_ohe = one_hot_hour_period(df)
display(df_ohe[["datetime", "hour_period", "period_вечір", "period_день", "period_ніч", "period_ранок"]].head(10))